# Experiment 12 Enhanced — Theme Business Needs + Stage + L3 — Enhanced Prompt, IDs Only

Same fixed-50 data and execution as E12; only the decision prompt is changed to treat Stage as a boundary rather than positive evidence.

## What the LLM sees

```text
task
theme
  └─ business_needs
value_stream_stage
  ├─ stage_id
  ├─ stage_name
  ├─ stage_description
  ├─ entrance_criteria
  └─ exit_criteria
candidate_l3_capabilities[]
  ├─ capability_id
  ├─ capability_name
  ├─ capability_description
  └─ capability_tier
```

**Not sent to the LLM:** Theme description, Epic description, Epic success criteria, L1/L2 hierarchy, ground truth, model reasons.


## Configuration and imports

In [ ]:
from pathlib import Path
from time import perf_counter
import ast
import json
import os

import pandas as pd
from IPython.display import display

from common import (
    call_llm_with_metrics,
    load_gateway,
    parse_json_response,
    save_results_excel,
    score_sets,
)

NOTEBOOK_DIR = Path.cwd()


def resolve_data_path(relative_path: str, env_var: str) -> Path:
    """Resolve a data file from an override, notebook folder, or repo root."""
    override = os.getenv(env_var)
    if override:
        return Path(override).expanduser()

    relative_path = Path(relative_path)
    search_roots = [
        NOTEBOOK_DIR,
        NOTEBOOK_DIR.parent,
        NOTEBOOK_DIR / "l3_experiments",
    ]
    seen = set()
    attempted = []

    for root in search_roots:
        candidate = root / relative_path
        key = str(candidate.resolve(strict=False))
        if key in seen:
            continue
        seen.add(key)
        attempted.append(candidate)
        if candidate.exists():
            return candidate

    tried = "\n  - ".join(str(path) for path in attempted)
    raise FileNotFoundError(
        f"Could not find {relative_path}. Tried:\n  - {tried}"
    )


PARQUET_PATH = resolve_data_path("full_golden.parquet", "L3_FULL_GOLDEN_PATH")
STAGE_PATH = resolve_data_path("VSSrv.csv", "L3_STAGE_PATH")
STAGE_CAPABILITY_MAP_PATH = resolve_data_path(
    "VSSCaprv (1).csv",
    "L3_STAGE_CAPABILITY_MAP_PATH",
)
GROUND_TRUTH_PATH = resolve_data_path(
    "results/epic_l3_ground_truth_full_golden.xlsx",
    "L3_GROUND_TRUTH_PATH",
)

SAMPLE_SIZE = 50
SAMPLE_SEED = 42

# Optional single-example inspection. Leave None for batch execution only.
INSPECTION_THEME_ID = None
INSPECTION_EPIC_KEY = None

EXPERIMENT_NAME = "E12_BUSINESS_NEEDS_STAGE_ENHANCED_PROMPT"


## Retrieval

In [ ]:
def clean_text(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return str(value).strip()


def parse_list_value(value) -> list[str]:
    if value is None:
        return []

    if hasattr(value, "tolist") and not isinstance(value, (str, bytes)):
        value = value.tolist()

    if isinstance(value, (list, tuple, set)):
        return [clean_text(item) for item in value if clean_text(item)]

    try:
        if pd.isna(value):
            return []
    except (TypeError, ValueError):
        pass

    text = str(value).strip()
    if not text:
        return []

    try:
        parsed = ast.literal_eval(text)
    except (SyntaxError, ValueError):
        return [text]

    if isinstance(parsed, (list, tuple, set)):
        return [clean_text(item) for item in parsed if clean_text(item)]
    return [clean_text(parsed)] if clean_text(parsed) else []


def read_table(path, *, sheet_name=None):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(
            path,
            dtype=str,
            encoding="cp1252",
            encoding_errors="replace",
        )
    return pd.read_excel(path, sheet_name=sheet_name, dtype=str)


def load_evaluation_population():
    population = read_table(
        GROUND_TRUTH_PATH,
        sheet_name="evaluation_population",
    )
    required = {
        "theme_key",
        "epic_key",
        "stage_ids",
        "gt_l3_ids",
        "candidate_l3_ids",
    }
    missing = required.difference(population.columns)
    if missing:
        raise KeyError(
            f"evaluation_population is missing columns: {sorted(missing)}"
        )

    population = (
        population
        .drop_duplicates(subset=["theme_key", "epic_key"], keep="first")
        .sort_values(["theme_key", "epic_key"], kind="stable")
        .reset_index(drop=True)
    )

    if population["epic_key"].duplicated().any():
        raise ValueError("evaluation_population contains duplicate Epic keys.")

    if len(population) < SAMPLE_SIZE:
        raise ValueError(
            f"Need {SAMPLE_SIZE} valid Epics, found only {len(population)}."
        )

    sample = population.sample(
        n=SAMPLE_SIZE,
        random_state=SAMPLE_SEED,
        replace=False,
    )
    return sample.sort_values(
        ["theme_key", "epic_key"],
        kind="stable",
    ).reset_index(drop=True)


evaluation_population = load_evaluation_population()
selected_pairs = set(
    zip(
        evaluation_population["theme_key"],
        evaluation_population["epic_key"],
    )
)
selected_theme_ids = set(evaluation_population["theme_key"])


def load_themes():
    frame = read_table(PARQUET_PATH)
    required = {"key", "description", "businessNeeds", "epic_keys"}
    missing = required.difference(frame.columns)
    if missing:
        raise KeyError(
            f"full_golden.parquet is missing columns: {sorted(missing)}"
        )

    themes = {}
    found_pairs = set()

    for _, row in frame.iterrows():
        theme_id = clean_text(row.get("key"))
        if theme_id not in selected_theme_ids:
            continue

        selected_epics = []
        for epic_key in parse_list_value(row.get("epic_keys")):
            if (theme_id, epic_key) in selected_pairs:
                selected_epics.append({"key": epic_key})
                found_pairs.add((theme_id, epic_key))

        if not selected_epics:
            continue

        themes[theme_id] = {
            "theme_description": clean_text(row.get("description")),
            "theme_business_needs": clean_text(row.get("businessNeeds")),
            "epics": selected_epics,
        }

    missing_pairs = selected_pairs - found_pairs
    if missing_pairs:
        raise ValueError(
            "Selected Theme/Epic pairs were not found in full_golden.parquet: "
            f"{sorted(missing_pairs)}"
        )

    return themes


themes = load_themes()
stage_frame = read_table(STAGE_PATH)
stage_capability_map = read_table(STAGE_CAPABILITY_MAP_PATH)


def stage_context(stage_id):
    match = stage_frame.loc[
        stage_frame["Value Stream Stage ID"].astype(str).str.strip() == stage_id
    ]
    if match.empty:
        raise KeyError(f"No stage metadata for {stage_id}")
    row = match.iloc[0]
    return {
        "stage_id": stage_id,
        "stage_name": clean_text(row["Value Stream Stage Name"]),
        "stage_description": clean_text(row["Value Stream Stage Description"]),
        "entrance_criteria": clean_text(row["Value Stream Stage Entrance Criteria"]),
        "exit_criteria": clean_text(row["Value Stream Stage Exit Criteria"]),
    }


print(
    f"Selected {len(evaluation_population)} valid Epics "
    f"with seed={SAMPLE_SEED} across {len(themes)} Themes."
)
display(evaluation_population.head(50))


## Candidate construction

In [ ]:
def candidate_rows_for_stage(stage_id):
    rows = stage_capability_map.loc[
        stage_capability_map["Value Stream Stage ID"].astype(str).str.strip()
        == stage_id
    ].copy()
    rows = (
        rows.drop_duplicates(subset=["Capability ID"], keep="first")
        .sort_values(["Capability Name", "Capability ID"], kind="stable")
    )

    return [
        {
            "capability_id": clean_text(row["Capability ID"]),
            "capability_name": clean_text(row["Capability Name"]),
            "capability_description": clean_text(row["Capability Description"]),
            "capability_tier": clean_text(row["Capability Tier"]),
        }
        for _, row in rows.iterrows()
    ]


## Production prompt

In [ ]:
SYSTEM_PROMPT = 'You are performing Level 3 business capability classification for one Epic.\n\nAn L3 capability is a Level 3 business capability: a specific business function within the enterprise capability hierarchy.\n\nUse Theme Business Needs as the business evidence.\nUse Value Stream Stage only to identify the relevant business boundary and candidate space.\n\nIMPORTANT\n\nA candidate must be supported by the Theme Business Needs.\nValue Stream Stage membership alone must never justify a selection.\n\nEVIDENCE\n\nTheme Business Needs describes the business outcomes and functions the Theme is intended to address.\n\nValue Stream Stage identifies which portion of those Business Needs is relevant to this Epic.\n\nFor each candidate L3:\n- capability_id is the exact identifier to return when selected.\n- capability_description is the primary definition of the business function.\n- capability_name is a supporting label.\n- capability_tier is taxonomy context only.\n\nDo not infer meaning from capability_id.\n\nCLASSIFICATION\n\n1. Identify the parts of Theme Business Needs that fall within the supplied Value Stream Stage boundary.\n2. Ignore Theme needs that belong outside that Stage.\n3. Compare the remaining business evidence against each candidate\'s capability_description.\n4. Select a candidate only when the Business Needs directly support that business function.\n\nDo not select a capability merely because:\n- it belongs to the Stage,\n- it shares terminology,\n- it is related to another supported capability,\n- it is upstream or downstream,\n- it supplies data or supporting functionality,\n- it is generally relevant to the Theme.\n\nBefore returning the result, remove any capability that is supported only by Stage membership or general relatedness rather than Theme Business Needs.\n\nOnly return capability_id values from the supplied candidates.\n\nIf none are supported, return an empty list.\n\nOUTPUT\n\n{"l3":["CAP00000123","CAP00000456"]}\n\nReturn JSON only. Do not return reasons, explanations, Markdown, or additional fields.'

def build_user_prompt(theme, epic, stage, candidate_rows):
    payload = {
        "task": "Select directly supported L3 business capability IDs from the supplied candidates.",
        "theme": {"business_needs": theme["theme_business_needs"]},
        "value_stream_stage": stage,
        "candidate_l3_capabilities": candidate_rows,
    }
    return json.dumps(payload, ensure_ascii=False, indent=2)


## Prediction

In [ ]:
def validate_l3_id_response(payload, candidate_ids):
    if set(payload) != {"l3"}:
        raise ValueError("LLM response must contain exactly one top-level field: l3.")
    raw = payload.get("l3")
    if not isinstance(raw, list):
        raise ValueError("LLM response must contain an 'l3' list.")
    allowed = {str(value).strip() for value in candidate_ids if str(value).strip()}
    selected = []
    seen = set()
    for index, capability_id in enumerate(raw, start=1):
        if not isinstance(capability_id, str):
            raise ValueError(f"L3 selection #{index} must be a capability_id string.")
        capability_id = capability_id.strip()
        if not capability_id:
            raise ValueError(f"L3 selection #{index} is empty.")
        if capability_id not in allowed:
            raise ValueError(f"LLM selected {capability_id}, which is not a supplied candidate.")
        if capability_id in seen:
            raise ValueError(f"LLM returned duplicate capability_id {capability_id}.")
        seen.add(capability_id)
        selected.append(capability_id)
    return selected


def predict_for_stage(gateway, theme, epic, stage_id):
    stage = stage_context(stage_id)
    candidates = candidate_rows_for_stage(stage_id)
    if not candidates:
        return {
            "stage": stage,
            "candidates": [],
            "user_prompt": build_user_prompt(theme, epic, stage, []),
            "raw_response": None,
            "selections": [],
            "metrics": None,
        }

    user_prompt = build_user_prompt(theme, epic, stage, candidates)
    raw_response, metrics = call_llm_with_metrics(
        gateway,
        SYSTEM_PROMPT,
        user_prompt,
        reasoning_effort="low",
    )
    selections = validate_l3_id_response(
        parse_json_response(raw_response),
        [candidate["capability_id"] for candidate in candidates],
    )
    return {
        "stage": stage,
        "candidates": candidates,
        "user_prompt": user_prompt,
        "raw_response": raw_response,
        "selections": selections,
        "metrics": metrics,
    }


def metric_text(value):
    return "n/a" if value is None else str(value)


def summarize_llm_calls(call_metrics):
    successful = call_metrics.loc[call_metrics["status"] == "ok"].copy()

    def numeric(column):
        return pd.to_numeric(successful[column], errors="coerce").dropna()

    latency = numeric("latency_seconds")
    input_tokens = numeric("input_tokens")
    output_tokens = numeric("output_tokens")
    total_tokens = numeric("total_tokens")

    return pd.DataFrame([{
        "successful_calls": len(successful),
        "failed_calls": int((call_metrics["status"] == "error").sum()),
        "usage_reported_calls": len(total_tokens),
        "avg_latency_seconds": float(latency.mean()) if len(latency) else None,
        "p50_latency_seconds": float(latency.quantile(0.50)) if len(latency) else None,
        "p95_latency_seconds": float(latency.quantile(0.95)) if len(latency) else None,
        "avg_input_tokens": float(input_tokens.mean()) if len(input_tokens) else None,
        "avg_output_tokens": float(output_tokens.mean()) if len(output_tokens) else None,
        "avg_total_tokens": float(total_tokens.mean()) if len(total_tokens) else None,
        "total_input_tokens": int(input_tokens.sum()) if len(input_tokens) else None,
        "total_output_tokens": int(output_tokens.sum()) if len(output_tokens) else None,
        "total_tokens": int(total_tokens.sum()) if len(total_tokens) else None,
    }])


def run_predictions(preflight):
    eligible_rows = preflight.loc[preflight["evaluation_eligible"]].copy()
    prediction_rows = []
    call_rows = []

    call_columns = [
        "experiment",
        "theme_id",
        "epic_key",
        "stage_id",
        "candidate_count",
        "status",
        "latency_seconds",
        "input_tokens",
        "output_tokens",
        "total_tokens",
        "selected_count",
        "error",
    ]

    print(
        f"\nRunning {EXPERIMENT_NAME}: "
        f"{len(eligible_rows)} preflight-valid Epics only"
    )

    if eligible_rows.empty:
        return (
            pd.DataFrame(),
            pd.DataFrame(columns=call_columns),
        )

    gateway = load_gateway()

    for epic_index, row in enumerate(
        eligible_rows.to_dict(orient="records"),
        start=1,
    ):
        theme_id = row["theme_id"]
        epic_key = row["epic_key"]
        stage_ids = json.loads(row["stage_ids"])
        theme = themes[theme_id]
        epic = next(
            item for item in theme["epics"] if item["key"] == epic_key
        )

        stage_predictions = []
        predicted_ids = set()
        status = "ok"
        error = None

        print(
            f"\n[LLM {epic_index}/{len(eligible_rows)}] "
            f"{theme_id} | {epic_key}"
        )

        for stage_id in stage_ids:
            started = perf_counter()
            try:
                result = predict_for_stage(
                    gateway,
                    theme,
                    epic,
                    stage_id,
                )
                candidates = result["candidates"]

                if not candidates:
                    print(f"  {stage_id} SKIP | no candidates")
                    continue

                metrics = result["metrics"]
                selected_ids = result["selections"]
                print(
                    f"  {stage_id} OK"
                    f" | candidates={len(candidates)}"
                    f" | latency={metrics['latency_seconds']:.3f}s"
                    f" | input_tokens={metric_text(metrics['input_tokens'])}"
                    f" | output_tokens={metric_text(metrics['output_tokens'])}"
                    f" | total_tokens={metric_text(metrics['total_tokens'])}"
                    f" | selected={selected_ids}"
                )

                call_rows.append({
                    "experiment": EXPERIMENT_NAME,
                    "theme_id": theme_id,
                    "epic_key": epic_key,
                    "stage_id": stage_id,
                    "candidate_count": len(candidates),
                    "status": "ok",
                    "latency_seconds": metrics["latency_seconds"],
                    "input_tokens": metrics["input_tokens"],
                    "output_tokens": metrics["output_tokens"],
                    "total_tokens": metrics["total_tokens"],
                    "selected_count": len(selected_ids),
                    "error": None,
                })
                stage_predictions.append({
                    "stage_id": stage_id,
                    "selected_l3_ids": selected_ids,
                })
                predicted_ids.update(selected_ids)
            except Exception as exc:
                latency = perf_counter() - started
                status = "error"
                error = str(exc)
                print(
                    f"  {stage_id} ERROR"
                    f" | latency={latency:.3f}s"
                    f" | {error}"
                )
                call_rows.append({
                    "experiment": EXPERIMENT_NAME,
                    "theme_id": theme_id,
                    "epic_key": epic_key,
                    "stage_id": stage_id,
                    "candidate_count": None,
                    "status": "error",
                    "latency_seconds": latency,
                    "input_tokens": None,
                    "output_tokens": None,
                    "total_tokens": None,
                    "selected_count": None,
                    "error": error,
                })
                break

        prediction_rows.append({
            "experiment": EXPERIMENT_NAME,
            "theme_id": theme_id,
            "epic_key": epic_key,
            "stage_ids": row["stage_ids"],
            "ground_truth_l3_ids": row["ground_truth_l3_ids"],
            "available_candidate_l3_ids": row["available_candidate_l3_ids"],
            "gt_found_in_candidates": row["gt_found_in_candidates"],
            "gt_missing_from_candidates": row["gt_missing_from_candidates"],
            "predicted_l3_ids": json.dumps(sorted(predicted_ids)),
            "stage_predictions": json.dumps(
                stage_predictions,
                ensure_ascii=False,
            ),
            "status": status,
            "error": error,
        })

    return (
        pd.DataFrame(prediction_rows),
        pd.DataFrame(call_rows, columns=call_columns),
    )


## Single-example inspection

In [ ]:
if INSPECTION_THEME_ID and INSPECTION_EPIC_KEY:
    match = evaluation_population.loc[
        (evaluation_population["theme_key"] == INSPECTION_THEME_ID)
        & (evaluation_population["epic_key"] == INSPECTION_EPIC_KEY)
    ]
    if match.empty:
        raise ValueError(
            "Inspection Theme/Epic is not in the selected 50-Epic population."
        )

    theme = themes[INSPECTION_THEME_ID]
    epic = next(
        item
        for item in theme["epics"]
        if item["key"] == INSPECTION_EPIC_KEY
    )
    stage_id = json.loads(match.iloc[0]["stage_ids"])[0]
    stage = stage_context(stage_id)
    candidates = candidate_rows_for_stage(stage_id)
    user_prompt = build_user_prompt(theme, epic, stage, candidates)

    print("SYSTEM PROMPT")
    print(SYSTEM_PROMPT)
    print()
    print("USER PROMPT")
    print(user_prompt)
    print()
    print("CANDIDATES")
    display(pd.DataFrame(candidates))

    if candidates:
        result = predict_for_stage(
            load_gateway(),
            theme,
            epic,
            stage_id,
        )
        print()
        print("MODEL RESPONSE")
        print(result["raw_response"])
        print()
        print("CALL METRICS")
        display(pd.DataFrame([result["metrics"]]))
    else:
        print()
        print("No candidates for this Stage; LLM call skipped.")
else:
    print(
        "Set INSPECTION_THEME_ID and INSPECTION_EPIC_KEY "
        "to inspect one of the selected 50 Epics."
    )


## Batch population, execution, and evaluation

The GT workbook already contains the prevalidated valid population. This experiment takes the same fixed random sample of 50 valid Epics (`SAMPLE_SEED = 42`) and never sends GT to the LLM.

In [ ]:
def preflight_population():
    rows = []

    for record in evaluation_population.to_dict(orient="records"):
        theme_id = clean_text(record["theme_key"])
        epic_key = clean_text(record["epic_key"])
        stage_ids = json.loads(record["stage_ids"])
        gt_ids = set(json.loads(record["gt_l3_ids"]))
        candidate_ids = set(json.loads(record["candidate_l3_ids"]))

        rows.append(
            {
                "experiment": EXPERIMENT_NAME,
                "theme_id": theme_id,
                "epic_key": epic_key,
                "stage_ids": json.dumps(stage_ids),
                "ground_truth_l3_ids": json.dumps(sorted(gt_ids)),
                "available_candidate_l3_ids": json.dumps(sorted(candidate_ids)),
                "gt_found_in_candidates": json.dumps(sorted(gt_ids)),
                "gt_missing_from_candidates": json.dumps([]),
                "evaluation_eligible": True,
                "evaluation_exclusion_reason": "",
                "preflight_error": None,
            }
        )

    preflight = pd.DataFrame(rows)
    print()
    print("================ POPULATION SUMMARY ================")
    print(f"Sample seed: {SAMPLE_SEED}")
    print(f"Epics selected: {len(preflight)}")
    print(f"Themes represented: {preflight['theme_id'].nunique()}")
    print("All selected Epics were prevalidated in the GT workbook.")
    print("====================================================")
    return preflight


def evaluate_predictions(prediction_frame):
    out = []
    for row in prediction_frame.to_dict(orient="records"):
        pred = set(json.loads(row["predicted_l3_ids"]))
        gt = set(json.loads(row["ground_truth_l3_ids"]))

        if row["status"] == "error":
            metrics = {
                "exact_match": None,
                "precision": None,
                "recall": None,
                "f1": None,
                "predicted_count": len(pred),
                "truth_count": len(gt),
            }
        else:
            metrics = score_sets(pred, gt)

        row.update(metrics)
        out.append(row)

    return pd.DataFrame(out)


def evaluation_summary(results, preflight):
    scored = results.loc[results["exact_match"].notna()]

    summary = pd.DataFrame([{
        "scope": "fixed_50_valid_epics_seed_42",
        "evaluated_epics": len(scored),
        "exact_match_accuracy": (
            scored["exact_match"].mean() if len(scored) else 0.0
        ),
        "mean_precision": (
            scored["precision"].mean() if len(scored) else 0.0
        ),
        "mean_recall": (
            scored["recall"].mean() if len(scored) else 0.0
        ),
        "mean_f1": scored["f1"].mean() if len(scored) else 0.0,
    }])

    diagnostics = pd.DataFrame([{
        "sample_seed": SAMPLE_SEED,
        "sample_size": SAMPLE_SIZE,
        "themes_selected": preflight["theme_id"].nunique(),
        "total_epics": len(preflight),
        "preflight_valid_epics": len(preflight),
        "preflight_invalid_epics": 0,
        "llm_prediction_errors": int((
            results["status"] == "error"
        ).sum()) if len(results) else 0,
        "scored_epics": len(scored),
    }])

    return summary, diagnostics


preflight = preflight_population()
predictions, llm_calls = run_predictions(preflight)
results = evaluate_predictions(predictions)
summary, diagnostics = evaluation_summary(results, preflight)
llm_call_summary = summarize_llm_calls(llm_calls)

print()
print("Evaluation summary")
display(summary)
print()
print("Population diagnostics")
display(diagnostics)
print()
print("LLM latency / token summary")
display(llm_call_summary)
print()
print("Per-call LLM metrics")
display(llm_calls.head(50))
if len(results):
    display(results.head(50))

output_path = save_results_excel(
    results,
    EXPERIMENT_NAME,
    "results",
    extra_sheets={
        "evaluation_summary": summary,
        "preflight": preflight,
        "diagnostics": diagnostics,
        "llm_calls": llm_calls,
        "llm_call_summary": llm_call_summary,
        "evaluation_population": evaluation_population,
    },
)
print(f"Saved {output_path}")
